In [ ]:
import requests
import feedparser

print("STEP 0: Initializing data sources")

sources = [
    {
        "name": "promed",
        "type": "rss",
        "url": "https://promedmail.org/promed-posts/?format=rss"
    },
    {
        "name": "cdc_fluview",
        "type": "rss",
        "url": "https://tools.cdc.gov/api/v2/resources/media/403372.rss"
    },
    {
        "name": "disease_sh",
        "type": "api",
        "url": "https://disease.sh/v3/covid-19/countries"
    }
]

print("Sources loaded:", len(sources))


def match_alert(title, disease=None, location=None, pathogen=None):
    title = title.lower()
    matches = []

    if disease and disease.lower() in title:
        matches.append("disease")

    if location and location.lower() in title:
        matches.append("location")

    if pathogen and pathogen.lower() in title:
        matches.append("pathogen")

    return len(matches) > 0


def fetch_alerts(disease=None, location=None, pathogen=None):
    alerts = []

    print("\nSTEP 1: Querying alert feeds")

    for src in sources:
        print("\nSource:", src["name"])

        try:
            if src["type"] == "rss":
                feed = feedparser.parse(src["url"])
                print("RSS entries:", len(feed.entries))

                for entry in feed.entries:
                    if match_alert(entry.title, disease, location, pathogen):
                        alerts.append({
                            "source": src["name"],
                            "title": entry.title,
                            "link": entry.link
                        })

            elif src["type"] == "api":
                r = requests.get(src["url"], timeout=10)
                data = r.json()
                print("API records:", len(data))

                for item in data:
                    alerts.append({
                        "source": src["name"],
                        "location": item.get("country"),
                        "disease": "covid",
                        "cases": item.get("cases")
                    })

        except Exception as e:
            print("Error reading", src["name"], e)

    print("\nSTEP 2: Data collection complete")
    print("Total records:", len(alerts))

    return alerts


print("\nSTEP 3: Enter epidemiological query")

disease = input("Disease: ").strip() or None
location = input("Location: ").strip() or None
pathogen = input("Pathogen: ").strip() or None

alerts = fetch_alerts(disease, location, pathogen)

print("\n================ RESULT ================")

print("\nUser Query")
print("Disease :", disease)
print("Location:", location)
print("Pathogen:", pathogen)

print("\nReal Alerts From Surveillance Systems")

real_alerts = [a for a in alerts if "title" in a]

if real_alerts:
    for a in real_alerts[:5]:
        print("\nSource:", a["source"])
        print("Alert :", a["title"])
        print("Link  :", a["link"])
else:
    print("No alerts matched the query")

# -----------------------------------------
# Filtered API summary for the given location
# -----------------------------------------

print("\nAPI Summary (disease.sh COVID country data)")

# All disease.sh records
api_records = [a for a in alerts if a.get("source") == "disease_sh"]

# If user specified a location, filter by that country name (case-insensitive)
if location:
    api_records = [
        rec for rec in api_records
        if isinstance(rec.get("location"), str)
        and rec["location"].lower() == location.lower()
    ]

if api_records:
    print("Total countries (after filter):", len(api_records))
    for rec in api_records[:5]:
        print(f"- {rec.get('location')}: {rec.get('cases')} cases")
else:
    print("No API records available for the given location")

print("\n=======================================")
